# Module 23: Deployment Basics
## Lesson: Containerizing and Deploying ML APIs

This lesson covers the fundamentals of deploying ML models to production:
Docker containerization, FastAPI APIs, model serving patterns, CI/CD,
and monitoring. You will build and deploy a production-ready ML prediction API.

In [ ]:
import json
import time
import numpy as np
from dataclasses import dataclass
from typing import List, Optional, Dict, Any

print("All imports successful")

### 1. Docker Fundamentals

Docker packages applications with all dependencies into lightweight
containers. A Dockerfile defines the build steps. Key concepts:
- **Images**: read-only templates (built from Dockerfile)
- **Containers**: running instances of images
- **Layers**: each instruction creates a cacheable layer
- **Multi-stage builds**: separate build and runtime environments

In [ ]:
dockerfile_content = """
# Multi-stage build for smaller production image
FROM python:3.10-slim AS builder

WORKDIR /app
COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt

FROM python:3.10-slim

WORKDIR /app
COPY --from=builder /root/.local /root/.local
COPY . .

ENV PATH=/root/.local/bin:$PATH
ENV PYTHONUNBUFFERED=1

EXPOSE 8000

HEALTHCHECK --interval=30s --timeout=3s --retries=3 \\
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
"""
print("Dockerfile with multi-stage build:")
print(dockerfile_content)

### 2. Building a FastAPI Prediction API

FastAPI is a modern Python web framework ideal for ML APIs:
- Automatic OpenAPI/Swagger documentation
- Pydantic validation (request/response schemas)
- Async support for high concurrency
- Automatic type conversion and validation

In [ ]:
# Simulated FastAPI app (run in a real environment for interactive testing)
from pydantic import BaseModel, Field
from typing import Optional, List


class PredictionInput(BaseModel):
    features: List[float] = Field(..., description="Feature vector for prediction")
    model_version: Optional[str] = Field(default="latest")


class PredictionOutput(BaseModel):
    prediction: float
    probability: Optional[float] = None
    model_version: str
    processing_time_ms: float


class HealthOutput(BaseModel):
    status: str
    model_loaded: bool
    uptime_seconds: float


print("Pydantic schemas defined:")
print(f"  PredictionInput fields: {PredictionInput.model_fields.keys()}")
print(f"  PredictionOutput fields: {PredictionOutput.model_fields.keys()}")
print(f"  HealthOutput fields: {HealthOutput.model_fields.keys()}")

### 3. Model Serving Patterns

Two primary patterns for serving ML models:

**Real-time (Online) Serving**:
- HTTP API that returns predictions in <100ms
- Suitable for fraud detection, recommendations, personalization
- Requires low-latency model and infrastructure

**Batch Serving**:
- Scheduled jobs that score large datasets
- Suitable for customer scoring, report generation
- Can use complex models (higher latency is OK)
- Often run via Airflow/Prefect

In [ ]:
print("=", "Model Serving Pattern Decision Guide", "=" * 30)
print()
print("Q: Do you need predictions in <1 second?")
print("  YES -> Real-time API")
print("  NO  -> Batch serving")
print()
print("Q: How many predictions per day?")
print("  <10,000 -> Real-time API (simple)")
print("  10k-1M  -> Either (consider latency needs)")
print("  >1M     -> Batch serving (more efficient)")
print()
print("Q: What's the model complexity?")
print("  Simple (LR, RF) -> Real-time possible")
print("  Complex (DL, ensembles) -> Batch preferred")
print()
print("Decision Matrix:")
print("  Latency-sensitive + Simple model = Real-time API")
print("  Latency-sensitive + Complex model = Optimize model first")
print("  Not latency-sensitive + Any model = Batch serving")

### 4. Production Deployment with Gunicorn + Uvicorn

For production, run Uvicorn behind Gunicorn as a process manager:
- **Uvicorn**: ASGI server, handles async requests
- **Gunicorn**: WSGI/ASGI process manager, handles workers, signals
- **Workers**: Number of worker processes (2-4 x CPU cores)
- **No --reload**: Only in development

In [ ]:
print("Development:")
print("  uvicorn app.main:app --reload --host 0.0.0.0 --port 8000")
print()
print("Production:")
print("  gunicorn -w 4 -k uvicorn.workers.UvicornWorker app.main:app ")
print("          --bind 0.0.0.0:8000")
print("          --timeout 120")
print("          --access-logfile -")
print("          --error-logfile -")
print()
print("Worker calculation:")
print("  Recommended = (2 x CPU cores) + 1")
print("  Example: 4 CPU cores -> 9 workers")

### 5. CI/CD for ML Models

CI/CD for ML differs from traditional CI/CD in key ways:
1. **Model validation**: Test model performance metrics (not just code)
2. **Data validation**: Check for data drift before retraining
3. **Model registry**: Promote models through staging environments
4. **A/B testing**: Route traffic between model versions

In [ ]:
ci_cd_yaml = """
name: ML Deploy Pipeline

on:
  push:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - uses: actions/setup-python@v4
        with: { python-version: "3.10" }
      - run: pip install -r requirements.txt
      - run: pytest tests/ --cov=app/ --cov-fail-under=80
      - run: python scripts/validate_model.py  # Check model perf

  build-and-deploy:
    needs: test
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - run: docker build -t ml-api:${{ github.sha }} .
      - run: docker tag ml-api:${{ github.sha }} ghcr.io/myorg/ml-api:latest
      - run: echo "${{ secrets.GITHUB_TOKEN }}" | docker login ghcr.io -u ${{ github.actor }} --password-stdin
      - run: docker push ghcr.io/myorg/ml-api:latest
      - run: |
          ssh deploy@server "
            docker pull ghcr.io/myorg/ml-api:latest &&
            docker-compose up -d --force-recreate api
          "
"""
print("CI/CD Pipeline:")
print(ci_cd_yaml)

### 6. Monitoring Deployed Models

Monitoring is critical for catching model degradation:
- **Data drift**: Input feature distributions change
- **Concept drift**: Relationship between features and target changes
- **Performance degradation**: Model accuracy declines over time
- **Infrastructure issues**: Latency spikes, error rates

In [ ]:
def detect_feature_drift(reference: np.ndarray, current: np.ndarray,
                         threshold: float = 0.05) -> Dict[str, Any]:
    """Simple drift detection using KS test."""
    from scipy.stats import ks_2samp

    n_features = reference.shape[1]
    drift_results = {"drift_detected": False, "features": {}}

    for i in range(n_features):
        stat, p_value = ks_2samp(reference[:, i], current[:, i])
        feature_drift = bool(p_value < threshold)
        drift_results["features"][f"feature_{i}"] = {
            "ks_statistic": float(f"{stat:.4f}"),
            "p_value": float(f"{p_value:.4f}"),
            "drift_detected": feature_drift,
        }
        if feature_drift:
            drift_results["drift_detected"] = True

    return drift_results


# Simulate drift detection
np.random.seed(42)
ref_data = np.random.normal(0, 1, (1000, 3))
current_data = np.random.normal(0.3, 1.1, (1000, 3))  # slight shift

results = detect_feature_drift(ref_data, current_data, threshold=0.05)
print("Drift Detection Results:")
print(f"  Overall drift detected: {results['drift_detected']}")
for feat, info in results["features"].items():
    print(f"  {feat}: KS={info['ks_statistic']}, p={info['p_value']}, "
          f"drift={'YES' if info['drift_detected'] else 'NO'}")

### 7. docker-compose for Multi-Service Deployments

docker-compose orchestrates multiple containers. For ML APIs, common
services include: API server, Redis cache, monitoring, and databases.

In [ ]:
compose_yaml = """
version: "3.8"

services:
  api:
    build:
      context: .
      dockerfile: Dockerfile
    ports:
      - "8000:8000"
    environment:
      - MODEL_PATH=/app/models/model.pkl
      - LOG_LEVEL=INFO
      - REDIS_URL=redis://redis:6379/0
    volumes:
      - ./models:/app/models:ro
    depends_on:
      - redis
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
    restart: unless-stopped

  redis:
    image: redis:7-alpine
    ports:
      - "6379:6379"
    volumes:
      - redis_data:/data

  # Monitoring service (prometheus + grafana)
  # prometheus:
  #   image: prom/prometheus
  #   volumes:
  #     - ./monitoring/prometheus.yml:/etc/prometheus/prometheus.yml

volumes:
  redis_data:
"""
print("docker-compose.yml:")
print(compose_yaml)

### Summary

In this lesson you learned:
- Docker fundamentals: Dockerfile, images, containers, multi-stage builds
- Building FastAPI prediction APIs with Pydantic validation
- Model serving patterns: real-time vs batch
- Production deployment with Gunicorn + Uvicorn
- CI/CD pipelines for ML model deployment
- Monitoring deployed models for drift and degradation
- Using docker-compose for multi-service deployments
- ngrok for testing local deployments remotely

**Next**: Complete the exercises to practice deploying your own ML API.